# Agentic financial research

Multi-agent workflow for issuer research. You ask a question in plain language; the system resolves the company, gathers market facts through tools, and returns a structured research brief.

**Pipeline**
1. **Research agent (A)** — chooses which data tools to call (price, volatility, news, sentiment, web search) and writes a typed `DataBrief`.
2. **Critic agent (B)** — reviews the brief, may request one missing fact, then publishes a `FinalReport`.
3. **Memory** — follow-up questions on the same issuer reuse stored facts; tools run only when new data is needed.

**Example questions:** `what is the news for apple`, `latest AAPL price`, or a full research request such as analysing financial health, market sentiment, 90-day risks, and a vol-grounded hedge for a ticker.

**API keys:** set `OPENROUTER_API_KEY` (preferred) and/or `GROQ_API_KEY` in Colab Secrets or a local `.env` file. Do not paste keys into notebook cells.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = os.environ.get(
    "AGENTIC_WORKFLOW_REPO",
    "https://github.com/lavanblavan/Agentic_financial_Analyser.git",
)

def ensure_project() -> Path:
    if IN_COLAB:
        root = Path("/content/Agentic_financial_Analyser")
        if not (root / "src/config.py").exists():
            subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
        else:
            subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
        os.chdir(root)
    else:
        root = Path.cwd().resolve()
        if not (root / "src/config.py").exists():
            raise FileNotFoundError("Open this notebook from the Agentic_financial_Analyser repo root.")

    req = root / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

ROOT = ensure_project()
print("project root:", ROOT)
print("runtime:", "colab" if IN_COLAB else "local")

In [ ]:
from src.config import describe_env, load_settings, missing_key_help

settings = load_settings()
print(describe_env(settings))
if not settings.llm_ready:
    print(missing_key_help(settings))

## Environment and query

The next cell loads API settings, then resolves your question to a ticker. Set `PICK` to a preset key or replace `QUERY` with your own text. The preview lists each preset with its inferred **task mode** (which tools the research agent is expected to need).

In [ ]:
# Pick a preset or paste your own question below.
from src.example_questions import (
    FOLLOWUP_QUESTIONS,
    PRIMARY_QUESTIONS,
    DEFAULT_FOLLOWUP_A,
    DEFAULT_FOLLOWUP_B,
    DEFAULT_PRIMARY,
    get_followup,
    get_primary,
)
from src.task_profile import infer_task_profile
from src.ticker import parse_research_query

PICK = DEFAULT_PRIMARY  # e.g. "news_only", "price_only", "messy_summary", "small_cap_news"
QUERY = get_primary(PICK)

print("Available primary questions:", ", ".join(PRIMARY_QUESTIONS))
print()
for name, text in PRIMARY_QUESTIONS.items():
    mode = infer_task_profile(text)["mode"]
    print(f"  {name:22} mode={mode:14}  {text[:60]}{'…' if len(text) > 60 else ''}")

parsed = parse_research_query(QUERY)
profile = infer_task_profile(QUERY)
COMPANY = parsed["ticker"]
FOLLOWUP_A = get_followup(DEFAULT_FOLLOWUP_A)
FOLLOWUP_B = get_followup(DEFAULT_FOLLOWUP_B)

print("\n--- active query ---")
print("pick:", PICK)
print("question:", parsed["task"])
print("task_mode:", profile["mode"])
print("required tools:", profile["required"])
print(
    f"{parsed['subject']!r} → {parsed['ticker']}  "
    f"({parsed['name']}, via {parsed['resolved_via']}, extract {parsed['extracted_via']})"
)
print("research follow-up:", FOLLOWUP_A)
print("full-pipeline follow-up:", FOLLOWUP_B)
print("all follow-ups:", list(FOLLOWUP_QUESTIONS))

## Tool layer (sanity check)

Five LangChain tools wrap the data modules: price/technicals, realized volatility, headlines, LLM sentiment, and web search. This cell invokes two tools directly on the resolved ticker to confirm the data layer before the agents run.

In [ ]:
from src.tools import ALL_TOOLS, get_price_data, calculate_volatility

print("tools:", [t.name for t in ALL_TOOLS])
print(get_price_data.invoke({"ticker": COMPANY})[:500])
print(calculate_volatility.invoke({"ticker": COMPANY, "window_days": 30}))

## Research agent (A)

LangGraph loop: `agent → tools → agent → … → finalize`

The model reads your question, decides which tools are needed, and produces a `DataBrief`. `ask_agent_a()` adds session memory: if the new question refers to the same issuer, prior observations are reused and only missing facts are fetched.

In [ ]:
from src.agent_a import build_agent_a

agent_a = build_agent_a()
print(agent_a.get_graph().draw_mermaid())

In [ ]:
from src.agent_a import format_answer
from src.memory import ask_agent_a, describe_memory, relate

print("session:", describe_memory())
print("plan:", relate(QUERY))
result = ask_agent_a(QUERY)

print("from_memory:", result.get("from_memory"))
print("task_mode:", result.get("task_mode"))
print("tool call order:")
for i, step in enumerate(result.get("tool_calls") or [], start=1):
    print(f"  {i}. {step['tool']}  {step.get('args') or step}")
if result.get("missing_after_run"):
    print("still missing:", result["missing_after_run"])
print()
print(format_answer(result))

In [ ]:
print("\n--- structured DataBrief ---")
if result.get("brief"):
    from pprint import pprint
    pprint(result["brief"], sort_dicts=False)
else:
    print(result.get("final_text") or "(no brief)")

### Research follow-up

Ask a related question on the same issuer. The memory layer decides whether to answer from the stored brief or call gap tools (e.g. refresh price, fetch more news).

In [ ]:
# Change key: refresh_price | recall_headlines | extend_news | switch_after_apple
FOLLOWUP_A = get_followup(DEFAULT_FOLLOWUP_A)

print("plan:", relate(FOLLOWUP_A))
follow_a = ask_agent_a(FOLLOWUP_A)
print("from_memory:", follow_a.get("from_memory"))
print("need_tools:", follow_a.get("need_tools"))
print("reused:", follow_a.get("reused"))
print()
print(follow_a.get("followup_answer") or format_answer(follow_a))
print()
print("session:", describe_memory())

## Critic agent (B) and final report

Typed handoff: `DataBrief` → `CritiqueDecision` → optional tool fulfill → revised brief → `FinalReport`.

Agent B checks for missing 90-day volatility, thin headlines, generic risks, or a hedge that ignores vol term structure. It may request **one** additional fact, then publish. The run below reuses the research agent output above to avoid duplicate fetches.

In [ ]:
from src.agent_b import build_two_agent_graph

two_agent = build_two_agent_graph()
print(two_agent.get_graph().draw_mermaid())

In [ ]:
from src.agent_b import format_final_report, run_two_agents

workflow = run_two_agents(QUERY, agent_a_result=result)

print("task_mode:", workflow.get("agent_a", {}).get("task_mode") or result.get("task_mode"))
print("revisions:", workflow["revisions"])
print("critique trail:")
for i, item in enumerate(workflow["critiques"], start=1):
    kind = item.get("request_kind") or ("publish" if not item.get("need_more_data") else "request")
    print(f"  {i}. {kind}: {item.get('reason')}")
if workflow.get("extra_facts"):
    print("facts B requested:", list(workflow["extra_facts"]))

In [ ]:
print(format_final_report(workflow))
print("\n--- structured FinalReport ---")
from pprint import pprint
if workflow.get("report"):
    pprint(workflow["report"], sort_dicts=False)
else:
    print("(no report)")

### Full-pipeline follow-up

`ask()` uses both the stored brief and the published report. On the same issuer it answers from memory when possible; otherwise it fetches only the facts required by the new question.

In [ ]:
from src.memory import ask
from src.tools import get_price_data

print("session before follow-up:", describe_memory())
print("cached price call:")
print(get_price_data.invoke({"ticker": COMPANY})[:400])

# Change key: recall_hedge | recall_risks | refresh_price | extend_search
FOLLOWUP_B = get_followup(DEFAULT_FOLLOWUP_B)
print("plan:", relate(FOLLOWUP_B))
follow_b = ask(FOLLOWUP_B)
print("from_memory:", follow_b.get("from_memory"))
print("reused:", follow_b.get("reused"))
print("need_tools:", follow_b.get("need_tools"))
print()
print(follow_b.get("followup_answer") or follow_b.get("report"))
print()
print("session after follow-up:", describe_memory())

## Observability

Every tool invocation is appended to `logs/agent_trace.jsonl` with timestamp, inputs, truncated output, duration, and whether the response came from cache.

In [ ]:
import pandas as pd
from src.tracing import read_trace

trace = read_trace(limit=20)
frame = pd.DataFrame(trace)
cols = [c for c in ["ts", "tool", "ok", "cached", "duration_ms", "inputs", "output"] if c in frame.columns]
display(frame[cols] if cols else frame)